## CSC3218 Deep Learning - Project-Based Examination (CIFAR-10)

- **Student name**: Wangolo Bachawa
- **Reg no**: M23B23/042
- **Access no**: B21303

This notebook follows the question paper structure:

- **Part One (Practical tasks)**
  - Task 1: Data preparation
  - Task 2: CNN model design
  - Task 3: Model training
  - Task 4: Model evaluation


### How to run
- Run cells **top to bottom**.
- The best model is saved as `cifar10_cnn_best.pt`.
- Figures are saved as PNG files.



In [1]:
import gc
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

# Reproducibility
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("PyTorch:", torch.__version__)

Device: cpu
PyTorch: 2.11.0+cpu


---
## Part One - Task 1: Data preparation

### Requirements
- Load the CIFAR-10 dataset
- Normalize pixel values
- Split into **train / validation / test**
- Apply **data augmentation** (random crop, horizontal flip; rotation optional)

### Why data augmentation is useful
CIFAR-10 images are small (32×32) and the training set is finite. Augmentation increases the variety of training examples using label-preserving transforms (crops, flips, small rotations). This encourages the network to learn **more general features** and typically reduces overfitting, improving validation/test performance.

In [2]:
# Paths — notebook assumes it lives in the same folder as "CIFAR-10 dataset"
ROOT = Path(".")
DATA_ROOT = ROOT / "CIFAR-10 dataset"
TRAIN_IMG_DIR = DATA_ROOT / "train" / "train"
LABELS_CSV = DATA_ROOT / "trainLabels.csv"

assert LABELS_CSV.is_file(), f"Missing {LABELS_CSV}"
assert TRAIN_IMG_DIR.is_dir(), f"Missing {TRAIN_IMG_DIR}"

labels_df = pd.read_csv(LABELS_CSV)
labels_df.head()

,id,label
0,1,frog
1,2,truck
2,3,truck
3,4,deer
4,5,automobile


In [3]:
class_names = sorted(labels_df["label"].unique().tolist())
class_to_idx = {c: i for i, c in enumerate(class_names)}
idx_to_class = {i: c for c, i in class_to_idx.items()}
num_classes = len(class_names)
print("Classes (", num_classes, "):", class_names)

Classes ( 10 ): ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [4]:
class CIFAR10PNGDataset(Dataset):
    """Load CIFAR-10-style PNGs indexed by numeric id from trainLabels.csv."""

    def __init__(self, dataframe: pd.DataFrame, img_dir: Path, transform=None):
        self.ids = dataframe["id"].astype(int).values
        self.targets = np.array([class_to_idx[l] for l in dataframe["label"]], dtype=np.int64)
        self.img_dir = Path(img_dir)
        self.transform = transform

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx: int):
        img_id = int(self.ids[idx])
        path = self.img_dir / f"{img_id}.png"
        img = Image.open(path).convert("RGB")
        y = int(self.targets[idx])
        if self.transform is not None:
            img = self.transform(img)
        return img, y

In [5]:
# (b) Normalization: scale to [0,1] then standardize with CIFAR-10 channel statistics
cifar_mean = (0.4914, 0.4822, 0.4465)
cifar_std = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose(
    [
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(degrees=10),  
        transforms.ToTensor(),
        transforms.Normalize(cifar_mean, cifar_std),
    ]
)

eval_transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize(cifar_mean, cifar_std),
    ]
)

In [6]:
# (c) Stratified split: 80% train, 10% validation, 10% test
df_train, df_temp = train_test_split(
    labels_df,
    test_size=0.2,
    random_state=42,
    stratify=labels_df["label"],
)
df_val, df_test = train_test_split(
    df_temp,
    test_size=0.5,
    random_state=42,
    stratify=df_temp["label"],
)

print("Train:", len(df_train), "| Val:", len(df_val), "| Test:", len(df_test))

train_ds = CIFAR10PNGDataset(df_train, TRAIN_IMG_DIR, transform=train_transform)
train_eval_ds = CIFAR10PNGDataset(df_train, TRAIN_IMG_DIR, transform=eval_transform)
val_ds = CIFAR10PNGDataset(df_val, TRAIN_IMG_DIR, transform=eval_transform)
test_ds = CIFAR10PNGDataset(df_test, TRAIN_IMG_DIR, transform=eval_transform)

# Batch size: use a smaller value on CPU to avoid memory issues
BATCH_SIZE = 128 if torch.cuda.is_available() else 64
NUM_WORKERS = 0  # Windows: keep 0 for stability

pin = torch.cuda.is_available()

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=pin)
train_eval_loader = DataLoader(train_eval_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin)

Train: 40000 | Val: 5000 | Test: 5000


In [7]:
# Quick sanity check (optional): visualize a few training samples (denormalized)
# Keep this OFF for submission to avoid embedding large images in the notebook file.
SHOW_SANITY_PLOT = False

def denormalize(tensor: torch.Tensor) -> torch.Tensor:
    mean = torch.tensor(cifar_mean).view(3, 1, 1)
    std = torch.tensor(cifar_std).view(3, 1, 1)
    return tensor * std + mean

if SHOW_SANITY_PLOT:
    xs, ys = next(iter(train_loader))
    fig, axes = plt.subplots(2, 5, figsize=(12, 5))
    for ax, x, y in zip(axes.flat, xs[:10], ys[:10]):
        im = denormalize(x).clamp(0, 1).permute(1, 2, 0).numpy()
        ax.imshow(im)
        ax.set_title(idx_to_class[int(y)])
        ax.axis("off")
    plt.suptitle("Augmented training samples (denormalized)")
    plt.tight_layout()
    plt.savefig("exam_sanity_augmented_samples.png", dpi=150)
    plt.close(fig)

---
## Part One - Task 2: CNN model design

### Minimum components 
- Convolution layers
- Activation functions
- Pooling layers
- Batch normalization
- Dropout
- Fully connected **or** Global Average Pooling
- Softmax output (used when converting logits to probabilities)

### Architecture choice (brief justification)
This CNN uses repeated **Conv → BatchNorm → ReLU** blocks with periodic **MaxPool**. Early layers learn low-level features (edges/textures) and deeper layers learn higher-level patterns. **BatchNorm** stabilizes training, **dropout** reduces overfitting, and **Global Average Pooling** keeps the classifier lightweight.

### Layer purpose summary
- **Conv2d**: learn feature maps
- **BatchNorm2d**: normalize activations per channel
- **ReLU**: non-linearity
- **MaxPool2d**: downsample and add translation tolerance
- **Dropout/Dropout2d**: regularization
- **AdaptiveAvgPool2d(1)**: global average pooling to a fixed-size vector
- **Linear**: map features to class logits
- **Softmax**: convert logits to probabilities for reporting/prediction display (training uses `CrossEntropyLoss`).

In [8]:
class CIFAR10CNN(nn.Module):
    """
    Purpose of main blocks:
    - Conv: learn spatial filters / feature maps.
    - BatchNorm: normalize activations per channel; smoother optimization.
    - ReLU: non-linearity.
    - MaxPool: spatial downsampling, translation tolerance, fewer parameters deeper in the net.
    - Dropout: randomly zero units during training to reduce co-adaptation (regularization).
    - AdaptiveAvgPool2d(1): global average pooling — fixed-size vector per channel.
    - Linear: map GAP features to class logits; softmax applied when probabilities are needed.
    """

    def __init__(self, num_classes: int = 10, dropout_fc: float = 0.5):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.25),
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.25),
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(nn.Dropout(dropout_fc), nn.Linear(256, num_classes))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.gap(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


model = CIFAR10CNN(num_classes=num_classes).to(device)
print(model)
dummy = torch.zeros(1, 3, 32, 32, device=device)
print("Output logits shape:", model(dummy).shape)

CIFAR10CNN(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU(inplace=True)
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (7): Dropout2d(p=0.2, inplace=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU(inplace=True)
    (11): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (12): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (13): ReLU(inplace=True)
    (14): MaxPool2d(kernel_size=2, stride=2, padding=0, dilat

---
## Part One - Task 3: Model training

### Framework
- PyTorch

### Training parameters
- Optimizer
- Learning rate
- Batch size
- Number of epochs

### Training strategies (used + brief justification)
- **Data augmentation**: improves generalization (Task 1)
- **Weight decay (AdamW)**: regularization to reduce overfitting
- **Learning-rate scheduling (cosine annealing)**: smooth LR decay for better convergence
- **Early stopping**: stops when validation loss stops improving, saving time
- **Mixed precision (AMP, if GPU available)**: speeds up training and reduces memory use

In [9]:
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Faster + more stable conv performance on GPU
torch.backends.cudnn.benchmark = torch.cuda.is_available()

USE_AMP = torch.cuda.is_available()  # mixed precision speeds up on GPU + lowers memory

LR = 3e-3
MAX_EPOCHS = 25  # keep runtime reasonable; should still reach strong accuracy
WEIGHT_DECAY = 1e-4
EARLY_STOP_PATIENCE = 5
EARLY_STOP_MIN_DELTA = 1e-4

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# Simple, strong default schedule for CIFAR-style training
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS, eta_min=1e-5)

In [10]:
import gc
from pathlib import Path
from contextlib import nullcontext

# Try to free any leftover objects before training
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)


def run_epoch(loader, model, criterion, optimizer=None):
    train_mode = optimizer is not None
    model.train(train_mode)

    total_loss = 0.0
    correct = 0
    total = 0

    autocast_ctx = torch.cuda.amp.autocast if USE_AMP else nullcontext

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        if train_mode:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train_mode):
            with autocast_ctx():
                logits = model(xb)
                loss = criterion(logits, yb)

            if train_mode:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        total_loss += float(loss.detach().cpu()) * xb.size(0)
        pred = logits.detach().argmax(dim=1)
        correct += int((pred == yb).sum().item())
        total += xb.size(0)

    return total_loss / total, correct / total


history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val = float("inf")
epochs_no_improve = 0
best_ckpt_path = "cifar10_cnn_best.pt"

for epoch in range(1, MAX_EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, model, criterion, optimizer)
    va_loss, va_acc = run_epoch(val_loader, model, criterion, optimizer=None)
    scheduler.step()

    history["train_loss"].append(tr_loss)
    history["val_loss"].append(va_loss)
    history["train_acc"].append(tr_acc)
    history["val_acc"].append(va_acc)

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
        f"val loss {va_loss:.4f} acc {va_acc:.4f} | lr {optimizer.param_groups[0]['lr']:.2e}"
    )

    # Save best model to DISK to avoid RAM spikes from cloning full state_dict every epoch
    if va_loss < best_val - EARLY_STOP_MIN_DELTA:
        best_val = va_loss
        torch.save(
            {"model": model.state_dict(), "class_names": class_names, "class_to_idx": class_to_idx},
            best_ckpt_path,
        )
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= EARLY_STOP_PATIENCE:
            print(
                f"Early stopping at epoch {epoch} (val loss did not beat best by "
                f"{EARLY_STOP_MIN_DELTA} for {EARLY_STOP_PATIENCE} epochs)."
            )
            break

# Reload best weights (if any were saved)
if Path(best_ckpt_path).exists():
    ckpt = torch.load(best_ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model"]) 
    print(f"Saved best weights to {best_ckpt_path}")
else:
    torch.save({"model": model.state_dict(), "class_names": class_names, "class_to_idx": class_to_idx}, best_ckpt_path)
    print(f"Saved weights to {best_ckpt_path}")

C:\Users\USER\AppData\Local\Temp\ipykernel_13608\2461172777.py:10: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)


Epoch 01/25 | train loss 1.9222 acc 0.3239 | val loss 1.6967 acc 0.4288 | lr 2.99e-03
Epoch 02/25 | train loss 1.6506 acc 0.4686 | val loss 1.4432 acc 0.5668 | lr 2.95e-03
Epoch 03/25 | train loss 1.5132 acc 0.5446 | val loss 1.3135 acc 0.6370 | lr 2.90e-03
Epoch 04/25 | train loss 1.4245 acc 0.5897 | val loss 1.2878 acc 0.6552 | lr 2.82e-03
Epoch 05/25 | train loss 1.3595 acc 0.6247 | val loss 1.1780 acc 0.7150 | lr 2.71e-03
Epoch 06/25 | train loss 1.3017 acc 0.6535 | val loss 1.1173 acc 0.7396 | lr 2.59e-03
Epoch 07/25 | train loss 1.2449 acc 0.6818 | val loss 1.1002 acc 0.7484 | lr 2.46e-03
Epoch 08/25 | train loss 1.2072 acc 0.7042 | val loss 1.0285 acc 0.7930 | lr 2.31e-03
Epoch 09/25 | train loss 1.1755 acc 0.7196 | val loss 1.0269 acc 0.7878 | lr 2.14e-03
Epoch 10/25 | train loss 1.1411 acc 0.7369 | val loss 0.9902 acc 0.8072 | lr 1.97e-03
Epoch 11/25 | train loss 1.1105 acc 0.7523 | val loss 0.9886 acc 0.8210 | lr 1.79e-03
Epoch 12/25 | train loss 1.0887 acc 0.7618 | val loss 

---
## Part One - Task 4: Model evaluation

### Metrics
- Training accuracy
- Validation accuracy
- Test accuracy
- Test precision, recall, F1 score

### Required outputs
- Loss curves and accuracy curves
- Confusion matrix
- Sample predictions



In [11]:
def collect_predictions(loader, model):
    model.eval()
    ys, yhats = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device, non_blocking=True)
            logits = model(xb)
            pred = logits.argmax(dim=1).cpu().numpy()
            ys.append(yb.numpy())
            yhats.append(pred)
    return np.concatenate(ys), np.concatenate(yhats)


# Training-set accuracy on original (non-augmented) images — standard for reporting
train_y, train_pred = collect_predictions(train_eval_loader, model)
val_y, val_pred = collect_predictions(val_loader, model)
test_y, test_pred = collect_predictions(test_loader, model)

train_acc = accuracy_score(train_y, train_pred)
val_acc = accuracy_score(val_y, val_pred)
test_acc = accuracy_score(test_y, test_pred)

prec_macro = precision_score(test_y, test_pred, average="macro", zero_division=0)
rec_macro = recall_score(test_y, test_pred, average="macro", zero_division=0)
f1_macro = f1_score(test_y, test_pred, average="macro", zero_division=0)

print(f"Train accuracy:   {train_acc:.4f}")
print(f"Validation acc:   {val_acc:.4f}")
print(f"Test accuracy:    {test_acc:.4f}")
print(f"Test precision (macro): {prec_macro:.4f}")
print(f"Test recall (macro):    {rec_macro:.4f}")
print(f"Test F1 (macro):        {f1_macro:.4f}")
print("\nPer-class report (test set):")
print(classification_report(test_y, test_pred, target_names=class_names, digits=4))

Train accuracy:   0.8974
Validation acc:   0.8686
Test accuracy:    0.8588
Test precision (macro): 0.8583
Test recall (macro):    0.8588
Test F1 (macro):        0.8581

Per-class report (test set):
              precision    recall  f1-score   support

    airplane     0.8760    0.8760    0.8760       500
  automobile     0.9404    0.9460    0.9432       500
        bird     0.8071    0.8200    0.8135       500
         cat     0.7494    0.6640    0.7041       500
        deer     0.8622    0.8260    0.8437       500
         dog     0.7407    0.8000    0.7692       500
        frog     0.8762    0.8780    0.8771       500
       horse     0.8854    0.8960    0.8907       500
        ship     0.9399    0.9380    0.9389       500
       truck     0.9060    0.9440    0.9246       500

    accuracy                         0.8588      5000
   macro avg     0.8583    0.8588    0.8581      5000
weighted avg     0.8583    0.8588    0.8581      5000



In [12]:
# Curves (save to disk; avoid embedding big images into the notebook)
epochs_ran = len(history["train_loss"])
fig = plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(range(1, epochs_ran + 1), history["train_loss"], label="Train loss")
plt.plot(range(1, epochs_ran + 1), history["val_loss"], label="Val loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss curves")
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(range(1, epochs_ran + 1), history["train_acc"], label="Train acc")
plt.plot(range(1, epochs_ran + 1), history["val_acc"], label="Val acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy curves")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("exam_curves_loss_acc.png", dpi=150)
plt.close(fig)

In [13]:
# Confusion matrix (save to disk; avoid embedding into the notebook)
cm = confusion_matrix(test_y, test_pred)
fig = plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=False, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion matrix (test set)")
plt.tight_layout()
plt.savefig("exam_confusion_matrix.png", dpi=150)
plt.close(fig)

In [14]:
# Sample predictions (optional). Saves figure to disk; keep OFF to avoid embedding into the notebook.
SHOW_SAMPLE_PREDICTIONS = False

if SHOW_SAMPLE_PREDICTIONS:
    model.eval()
    xb, yb = next(iter(test_loader))
    xb = xb.to(device)
    with torch.no_grad():
        probs = F.softmax(model(xb), dim=1).cpu()
    xb_cpu = xb.cpu()

    fig, axes = plt.subplots(2, 5, figsize=(14, 6))
    for ax, img, t, p in zip(axes.flat, xb_cpu[:10], yb[:10], probs[:10]):
        im = denormalize(img).clamp(0, 1).permute(1, 2, 0).numpy()
        ax.imshow(im)
        pred_idx = int(p.argmax())
        title = f"True: {idx_to_class[int(t)]}\nPred: {idx_to_class[pred_idx]} ({p[pred_idx]:.2f})"
        ax.set_title(title, fontsize=8)
        ax.axis("off")
    plt.suptitle("Sample test predictions (softmax confidence)")
    plt.tight_layout()
    plt.savefig("exam_sample_predictions.png", dpi=150)
    plt.close(fig)

In [15]:
# Quick results summary (run after training)
from pathlib import Path

print('Best checkpoint:', Path('cifar10_cnn_best.pt').resolve())
if Path('cifar10_cnn_best.pt').exists():
    ckpt = torch.load('cifar10_cnn_best.pt', map_location=device)
    model.load_state_dict(ckpt['model'])

train_y, train_pred = collect_predictions(train_eval_loader, model)
val_y, val_pred = collect_predictions(val_loader, model)
test_y, test_pred = collect_predictions(test_loader, model)

train_acc = accuracy_score(train_y, train_pred)
val_acc = accuracy_score(val_y, val_pred)
test_acc = accuracy_score(test_y, test_pred)

prec_macro = precision_score(test_y, test_pred, average='macro', zero_division=0)
rec_macro = recall_score(test_y, test_pred, average='macro', zero_division=0)
f1_macro = f1_score(test_y, test_pred, average='macro', zero_division=0)

print(f'Train acc: {train_acc:.4f} | Val acc: {val_acc:.4f} | Test acc: {test_acc:.4f}')
print(f'Test precision/recall/F1 (macro): {prec_macro:.4f} / {rec_macro:.4f} / {f1_macro:.4f}')

print('Saved figures:')
for f in ['exam_curves_loss_acc.png','exam_confusion_matrix.png','exam_sample_predictions.png','exam_sanity_augmented_samples.png']:
    if Path(f).exists():
        print(' -', f, f'({Path(f).stat().st_size/1024:.1f} KB)')

Best checkpoint: E:\deep learning final exam\cifar10_cnn_best.pt
Train acc: 0.8974 | Val acc: 0.8686 | Test acc: 0.8588
Test precision/recall/F1 (macro): 0.8583 / 0.8588 / 0.8581
Saved figures:
 - exam_curves_loss_acc.png (66.7 KB)
 - exam_confusion_matrix.png (43.6 KB)
